<a href="https://colab.research.google.com/github/oktaforai-okta/ProGearSalesAI/blob/main/notebooks/progear-inventory-authorization-story.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Secure your custom AI agent with Okta
## From the ProGear business story to a reusable implementation

You already have an AI agent. It may use your own Python service, LangGraph, Amazon Bedrock, an MCP client, or another framework. This guide shows where **Okta for AI Agents** fits into that existing design.

ProGear is the worked example: one ProGear Sales Agent acts for Sarah or Mike and requests access to Inventory. The same pattern applies when you replace ProGear with your agent and Inventory with your protected API, MCP server, or business service.

### Choose your path

| If you are... | Read... | Run code? |
|---|---|---|
| Business or security leader | Parts A and B | No |
| Okta administrator or architect | Parts A through C, then E | Optional |
| Agent developer | Parts A, B, D, and E | Yes, safe local labs |

### Safety first

- Every default code cell is local or read-only.
- No cell contains a password, client secret, private key, authorization code, or live token.
- Do not paste credentials or tokens into notebook cells or outputs.
- The live token-exchange lab is a **template only** and is disabled by default.
- Use a secrets manager in production. Never send a token to an online decoder.

### Learning outcomes

By the end, you can explain the business decision, configure the Okta objects, place token exchange into an existing agent, validate the resource token, and troubleshoot common errors.


# Part A: Understand the authorization story

## 1. One agent, different permissions

ProGear employees use one conversational assistant: the **ProGear Sales Agent**. The employee does not need to know which internal service stores inventory.

| Employee | Job responsibility | Inventory access |
|---|---|---|
| **Sarah Sales** | Helps customers and checks availability | Can **read** inventory, but cannot **increase** it |
| **Mike Manager** | Manages warehouse stock | Can **read** and **increase** inventory |

Both use the same agent. The decision changes because Okta evaluates the signed-in employee, the agent identity, the resource, and the permission required by the action.


## 2. Read and write are separate permissions

A request to **check inventory** needs `inventory:read`. A request to **increase inventory** needs `inventory:write`.

| User and request | Permission needed | Policy result | User experience |
|---|---|---|---|
| Sarah checks stock | `inventory:read` | Allowed | Inventory can be shown |
| Sarah increases stock | `inventory:write` | Denied | Contact a manager for assistance or approval |
| Mike checks stock | `inventory:read` | Allowed | Inventory can be shown |
| Mike increases stock | `inventory:write` | Allowed | The request may continue to additional safeguards |

A denied write means the authorization control worked. It does not mean the AI agent failed.


## 3. Safe interactive policy simulator

Choose a person and action, then run the cell. This is a local simulation. It does not connect to Okta or change inventory.


In [ ]:
# @title Choose an employee and inventory action
employee = "Sarah Sales" # @param ["Sarah Sales", "Mike Manager"]
action = "Increase inventory" # @param ["Check inventory", "Increase inventory"]

APPROVED_DENIAL = (
    "I can’t increase inventory with your current permissions. "
    "Please contact your manager for assistance or approval."
)
POLICY = {
    "Sarah Sales": {"inventory:read"},
    "Mike Manager": {"inventory:read", "inventory:write"},
}
ACTION_TO_PERMISSION = {
    "Check inventory": "inventory:read",
    "Increase inventory": "inventory:write",
}
required_permission = ACTION_TO_PERMISSION[action]
allowed = required_permission in POLICY[employee]
print(f"Employee: {employee}")
print(f"Requested action: {action}")
print(f"Permission required: {required_permission}")
print(f"Decision: {'ALLOWED' if allowed else 'DENIED'}")
print()
print("The ProGear Sales Agent may continue with this request." if allowed else APPROVED_DENIAL)


Employee: Sarah Sales
Requested action: Increase inventory
Permission required: inventory:write
Decision: DENIED

I can’t increase inventory with your current permissions. Please contact your manager for assistance or approval.


### Expected result for Sarah's write request

```text
Employee: Sarah Sales
Requested action: Increase inventory
Permission required: inventory:write
Decision: DENIED

I can’t increase inventory with your current permissions. Please contact your manager for assistance or approval.
```

The response does not expose an internal service name, routing component, or policy implementation. Those details belong in administrator diagnostics and audit logs.


## 4. What happens behind the scenes

```text
Signed-in employee
      │ asks the customer-owned agent to perform an action
      ▼
Customer-owned AI agent
      │ identifies itself and the employee it represents
      ▼
Okta Org Authorization Server
      │ issues a short-lived ID-JAG binding user + agent + requested resource
      ▼
Custom Authorization Server
      │ evaluates user/group policy and requested scope
      ├─ allowed → scoped access token
      └─ denied  → no resource token
      ▼
Protected API, MCP server, or business service
```

**ID-JAG** is a short-lived signed proof that says: this governed agent is acting on behalf of this signed-in user. The final resource access token is separate and carries the approved resource permission.


# Part B: Map the pattern to your architecture

## 5. The reusable reference architecture

| Component | ProGear example | Replace with your world | Owner |
|---|---|---|---|
| User-facing client | ProGear web app | Your web, mobile, or internal client | Customer |
| AI runtime | ProGear Sales Agent backend | Your agent service or platform | Customer |
| Agent identity | ProGear workload principal | Your registered or imported AI agent | Okta + customer |
| Resource | Inventory service | Your API, MCP server, SaaS, or database service | Customer |
| Permissions | `inventory:read`, `inventory:write` | Your custom OAuth scopes | Customer |
| Policy groups | Sales, Warehouse | Your directory groups or users | Okta admin |
| Audit trail | ID-JAG and access-token grants | Your System Log evidence | Okta |

### One workload principal does not mean one code module

Your application may have many tools, prompts, skills, or internal workflow nodes. Okta governs the stable agent identity and its resource connections. Internal implementation components do not each need to become a user-facing agent.

### Direct exchange versus Agent Gateway

- **Direct token exchange:** your calling application or agent runtime performs the two exchanges and calls the protected resource. This is the runnable production pattern in this guide.
- **Agent Gateway:** uses the same registered agent identity to mediate and attribute tool calls, but Agent Gateway is currently a research-release capability and is not a generally available self-service integration path. Engage your Okta team if you are participating in that program.

The Sarah/Mike policy story applies to the same governed identity and resource policies. The executable sections in this notebook use direct token exchange.


## 6. Fill in your integration worksheet

Edit only the placeholder strings. Do not enter secrets. This creates a planning summary, not an Okta configuration.


In [ ]:
# @title Map ProGear concepts to your environment
agent_name = "My Customer Agent" # @param {type:"string"}
resource_name = "My Protected API" # @param {type:"string"}
read_scope = "resource:read" # @param {type:"string"}
write_scope = "resource:write" # @param {type:"string"}
reader_group = "Resource-Readers" # @param {type:"string"}
writer_group = "Resource-Managers" # @param {type:"string"}

worksheet = {
    "agent": agent_name,
    "resource": resource_name,
    "permissions": {"read": read_scope, "write": write_scope},
    "policy": {reader_group: [read_scope], writer_group: [read_scope, write_scope]},
}
import json
print(json.dumps(worksheet, indent=2))


## 7. Decide where token exchange lives

Use this decision table before writing code.

| Your design | Recommended insertion point |
|---|---|
| Python or FastAPI agent | A reusable token-exchange module called after intent/scope selection |
| AWS Bedrock AgentCore | A reusable exchange module before invocation; pass the final bearer token in the platform's secure session context |
| MCP client calling protected servers | Resolve a token per resource connection before the tool call |
| Agent Gateway research program | If your Okta team has enabled the research release, follow that program's gateway-specific deployment guidance |
| Custom orchestrator | Exchange after choosing the minimum scope, before executing the action |

Do not exchange once at startup and reuse the token for every user. Cache only by **user + agent + resource + scope**, and only until token expiry.


# Part C: Configure Okta for a customer-owned agent

## 8. Prerequisites

- An Okta Identity Engine org subscribed to **Okta for AI Agents**
- An administrator permitted to manage AI agents, applications, groups, authorization servers, policies, and resource connections
- A customer-owned agent you can configure
- A protected resource and its OAuth audience
- Custom resource scopes, such as `inventory:read` and `inventory:write`
- At least two owners for the AI agent
- A production secrets manager for private key material

Keep **User access**, **Client registration**, **Machine access**, and **Resource connections** distinct. They solve different problems.


## 9. Register or import the AI agent

1. Go to **Directory → AI Agents**.
2. Import a supported third-party agent, or register a custom agent manually.
3. Add a clear name and description.
4. Assign at least two owners.
5. Copy the agent's client identifier for the agent builder.
6. Activate the agent after configuration is complete.

The registered object is a Workload Principal: the stable identity you manage, audit, certify, deactivate, and connect to resources.


## 10. Configure Client registration

For the production custom-agent pattern in this guide, use **Public/private key** (`private_key_jwt`). It is the supported live client-authentication method for the agent workload identity.

Some Agent Gateway research-release materials describe Client ID only and client-secret options. Do not design a production integration around those preview methods unless Okta confirms availability and support for your program and org.

Configure public/private key authentication:

1. Generate the key pair in your approved key-management process.
2. Register only the public JWK with the agent.
3. Store the private JWK in a secrets manager.
4. Keep the `kid` with the key metadata.
5. Implement short-lived RS256 client assertions in the agent runtime.

Never commit the private key or place it in a `NEXT_PUBLIC_` variable.


## 11. Configure direct User access

1. Open the AI agent's **User access** tab.
2. Select **Allow user access**.
3. Okta creates and permanently binds an OIDC app to the AI agent.
4. Configure callback and logout URLs for your client.
5. Assign the intended users or groups to the bound app.
6. Use the Org Authorization Server for the user ID token.

### Existing legacy agents

For an agent using the outdated User sign-on method, remove its old User access link and recreate User access. In the current migration UI, selecting an existing OIDC app may not be supported unless the agent is deleted and registered again. Capture assignments and configuration before making that change.


## 12. Create the protected resource boundary

For the ProGear Inventory example:

1. Create or select a **Custom Authorization Server**.
2. Set the resource audience, for example `api://your-inventory-api`.
3. Add custom scopes such as `inventory:read` and `inventory:write`.
4. Do not use `openid`, `profile`, or `email` as resource scopes. System scopes are not the downstream permissions for ID-JAG resource exchange.
5. Create an access policy assigned to the AI agent client.
6. Enable the JWT Bearer grant in the policy rule.
7. Add policy rules:
   - Sales group → `inventory:read`
   - Warehouse group → `inventory:read`, `inventory:write`
8. Keep rules active and request one least-privilege scope for the current action.

If any requested scope is not grantable, the exchange fails. Do not expect silent partial down-scoping.


## 13. Add the Authorization server resource connection

On the AI agent's **Resource connections** tab:

1. Add an **Authorization server** resource connection.
2. Select the Custom Authorization Server.
3. Enter the resource audience URL.
4. Choose **Only allow** / `INCLUDE_ONLY`.
5. Select only the custom scopes the agent may request.
6. Activate the connection.

The connection defines the maximum resource boundary for the agent. The authorization-server policy still decides whether the current user receives the requested scope.

### Machine access is different

Use **Machine access** only when non-human callers need to call the AI agent as a resource. It is not required for the Sarah/Mike user-delegation scenario.


## 14. Configuration checkpoint

Before writing exchange code, confirm:

- Agent status is Active and owners are correct.
- Client registration method matches what your runtime implements.
- The user is assigned to the direct User access app.
- The user ID token comes from the Org Authorization Server and is intended for the bound sign-on app.
- The Custom Authorization Server contains the custom scopes.
- The policy is assigned to the AI agent client and enables JWT Bearer.
- The policy rules reference the correct users/groups and scopes.
- The resource connection points to the same authorization server, audience, and allowed scopes.
- The public key registered in Okta matches the runtime private key `kid`.


# Part D: Wire Okta into your custom agent

## 15. The insertion point

Your existing agent likely already does this:

```python
intent = route(user_request)
result = call_business_resource(intent)
```

Add delegated authorization between routing and execution:

```python
intent = route(user_request)
required_scope = one_scope_for(intent)
access_token = exchange_user_identity(user_id_token, required_scope)
result = call_business_resource(intent, bearer_token=access_token)
```

A protected write never runs when token exchange is denied. Convert the denial into a useful business message, and keep the technical reason in logs.


## 16. Platform-neutral Python reference

The repository includes [`examples/token_exchange.py`](../examples/token_exchange.py). It exposes:

- `TokenExchangeConfig.from_environment()`
- `OktaTokenExchange.build_client_assertion()`
- `OktaTokenExchange.get_id_jag()`
- `OktaTokenExchange.get_access_token()`
- `OktaTokenExchange.verify_access_token()`

Dependencies:

```text
PyJWT[crypto]>=2.8.0
requests>=2.31.0
```

Production environment variable names:

```text
OKTA_DOMAIN
OKTA_CUSTOM_AS_ID
OKTA_SCOPE
OKTA_RESOURCE_AUDIENCE
OKTA_AGENT_CLIENT_ID
OKTA_AGENT_KEY_ID
OKTA_AGENT_PRIVATE_KEY_JWK   # secrets manager only
```

The client assertion is generated twice. Its `aud` must exactly match the token endpoint invoked in that step.


## 17. Safe lab: build a local client assertion

This lab creates an ephemeral RSA key in Colab. It never contacts Okta and never stores the key. It demonstrates the required client-assertion claims.


In [ ]:
import base64
import json
import time
import uuid
import jwt
from cryptography.hazmat.primitives.asymmetric import rsa

private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
public_numbers = private_key.public_key().public_numbers()
def b64url_int(value):
    raw = value.to_bytes((value.bit_length() + 7) // 8, "big")
    return base64.urlsafe_b64encode(raw).rstrip(b"=").decode()

kid = "ephemeral-lab-key"
endpoint = "https://example.okta.com/oauth2/v1/token"
now = int(time.time())
assertion = jwt.encode(
    {"iss": "example-agent-client", "sub": "example-agent-client", "aud": endpoint,
     "iat": now, "exp": now + 60, "jti": str(uuid.uuid4())},
    private_key, algorithm="RS256", headers={"kid": kid},
)
public_jwk = {"kty": "RSA", "kid": kid, "use": "sig", "alg": "RS256",
              "n": b64url_int(public_numbers.n), "e": b64url_int(public_numbers.e)}
claims = jwt.decode(assertion, key=jwt.PyJWK.from_dict(public_jwk).key,
                    algorithms=["RS256"], audience=endpoint)
print(json.dumps({k: claims[k] for k in ("iss", "sub", "aud", "iat", "exp", "jti")}, indent=2))
print("Signature verified locally with the public JWK.")


## 18. Safe lab: inspect both OAuth request shapes

The placeholders below are deliberately not tokens. The cell redacts assertion values before display.


In [ ]:
TOKEN_EXCHANGE_GRANT = "urn:ietf:params:oauth:grant-type:token-exchange"
JWT_BEARER_GRANT = "urn:ietf:params:oauth:grant-type:jwt-bearer"
CLIENT_ASSERTION_TYPE = "urn:ietf:params:oauth:client-assertion-type:jwt-bearer"

step_1 = {
    "grant_type": TOKEN_EXCHANGE_GRANT,
    "client_assertion_type": CLIENT_ASSERTION_TYPE,
    "client_assertion": "<signed-for-org-token-endpoint>",
    "subject_token": "<signed-in-user-id-token>",
    "subject_token_type": "urn:ietf:params:oauth:token-type:id_token",
    "requested_token_type": "urn:ietf:params:oauth:token-type:id-jag",
    "scope": "inventory:write",
    "audience": "https://example.okta.com/oauth2/example-custom-as",
}
step_2 = {
    "grant_type": JWT_BEARER_GRANT,
    "client_assertion_type": CLIENT_ASSERTION_TYPE,
    "client_assertion": "<signed-for-custom-as-token-endpoint>",
    "assertion": "<id-jag-from-step-1>",
}
print("STEP 1: Org Authorization Server")
print(json.dumps(step_1, indent=2))
print("\nSTEP 2: Custom Authorization Server")
print(json.dumps(step_2, indent=2))


## 19. Safe lab: validate an illustrative resource token

A real resource server must validate the signature through the authorization server's JWKS. This local lab additionally demonstrates issuer, audience, expiration, and scope checks using an ephemeral key.


In [ ]:
resource_private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
resource_public = resource_private_key.public_key().public_numbers()
issuer = "https://example.okta.com/oauth2/example-custom-as"
audience = "api://example-inventory"
now = int(time.time())
resource_token = jwt.encode(
    {"sub": "example-user", "act": {"sub": "example-agent"}, "iss": issuer,
     "aud": audience, "scp": ["inventory:read"], "iat": now, "exp": now + 300},
    resource_private_key, algorithm="RS256", headers={"kid": "resource-lab-key"},
)
resource_jwk = {"kty": "RSA", "kid": "resource-lab-key", "use": "sig", "alg": "RS256",
                "n": b64url_int(resource_public.n), "e": b64url_int(resource_public.e)}
verified = jwt.decode(resource_token, key=jwt.PyJWK.from_dict(resource_jwk).key,
                      algorithms=["RS256"], issuer=issuer, audience=audience,
                      options={"require": ["exp", "iss", "aud"]})
required_scope = "inventory:read"
assert required_scope in verified["scp"]
print(json.dumps({"subject": verified["sub"], "actor": verified["act"]["sub"],
                  "audience": verified["aud"], "scopes": verified["scp"],
                  "expires_in_seconds": verified["exp"] - int(time.time())}, indent=2))
print("Signature, issuer, audience, expiry, and scope validated locally.")


## 20. Optional live exchange template, disabled by default

A live exchange requires a fresh user ID token and the agent's private key. Those values must come from your client and secrets manager at runtime, not from notebook source.

The cell below is intentionally disabled. It prints no raw tokens. Advanced users may adapt it in an isolated test org after security review.


In [ ]:
# @title Optional live lab gate
ENABLE_LIVE_EXCHANGE = False # @param {type:"boolean"}
if not ENABLE_LIVE_EXCHANGE:
    print("Live exchange is disabled. The default notebook performs no authenticated calls.")
else:
    raise RuntimeError(
        "Implement this only in an isolated test org using Colab Secrets or your secrets manager. "
        "Do not paste tokens or private keys into the notebook."
    )


## 21. Call the protected resource

After Step 2 succeeds:

1. Validate the access token before trusting it.
2. Send it to the protected resource as `Authorization: Bearer <access_token>`.
3. Enforce the required scope again at the resource server.
4. Never forward the ID token or ID-JAG as the resource bearer token.
5. Never log any raw token.

For an MCP server, apply the same checks before executing a tool. If you participate in the Agent Gateway research program, configure the corresponding resource according to that program's gateway-specific guidance.


# Part E: Operate it safely

## 22. Production checklist

### Identity and access
- Agent has current owners and is Active.
- User assignments are reviewed on the direct User access app.
- Machine access is configured only if non-human callers need it.
- Workload access can be stopped by deactivating the agent.

### Credentials
- Private keys live in a secrets manager.
- The registered public JWK and runtime private JWK share the same `kid`.
- Rotation supports overlap, verification, and retirement of the old key.
- Client assertions are short-lived and use unique `jti` values.

### Least privilege
- Intent maps to one minimum scope.
- Resource connections use an explicit allow list.
- Custom AS rules grant scopes to the correct users/groups.
- One user's cached token is never reused for another user.

### Token handling
- Cache by user + agent + resource + scope only until `exp`.
- Validate signature, issuer, audience, expiry, and scope on every resource request.
- Sign out or revoke standard access tokens according to your application policy.
- Never expose raw tokens in UI, logs, URLs, screenshots, or support tickets.

### Audit
- Review `app.oauth2.token.grant.id_jag` for ID-JAG decisions.
- Review access-token grant events for the final resource token.
- Correlate user, agent, resource, requested scope, outcome, and transaction context.


## 23. Troubleshooting

| Error or symptom | Likely cause | Check |
|---|---|---|
| `invalid_scope` | System scope or scope missing from the resource connection / Custom AS | Use a custom resource scope and align both configurations |
| `invalid_token` in Step 1 | Expired ID token or wrong sign-on app | Sign in again; confirm the token is for the bound User access client |
| `invalid_grant` | Wrong assertion, expired one-time value, or wrong flow | Recreate fresh assertions and confirm grant type |
| `invalid_client: JWKSet not configured` | Public key not registered | Register the public JWK on the agent |
| `invalid_client: kid is invalid` | Runtime `kid` does not match Okta | Use the registered key ID |
| `no_matching_policy` / `access_denied` | Client, group, grant, or scope does not match an active rule | Check assigned client, JWT Bearer, user/group, and scope |
| Only service apps can use client credentials | Wrong client type or flow | Use the AI Agent client for token exchange |
| `invalid_audience` | Endpoint-specific assertion `aud` or Step 1 resource audience is wrong | Assertion `aud` is the token endpoint; Step 1 `audience` is the Custom AS issuer |
| Read request is denied | Agent asked for write or too many scopes | Route to exactly `inventory:read` |
| Group change appears stale | Existing sign-in session | Sign out fully and sign in again |

Keep the technical reason in administrator logs. Return a useful business message to the employee.


## 24. Test plan without changing business data

| Test | Expected result | Evidence |
|---|---|---|
| Sarah requests inventory read | Token with `inventory:read` | Successful ID-JAG and access-token grants |
| Sarah requests inventory write | No write token | Policy denial and user-friendly message |
| Mike requests inventory read | Token with `inventory:read` | Successful grants |
| Mike requests inventory write | Token with `inventory:write` | Successful grants; do not execute the actual write for this test |
| Token presented to wrong audience | Rejected | Resource validation failure |
| Token missing required scope | Rejected | Resource validation failure |

A complete security test proves both allowed and denied cases. Testing only the happy path is not enough.


## 25. Optional read-only demo health check

This checks only whether the public ProGear demo backend is available. It does not authenticate or change data.


In [ ]:
import json
from urllib.error import URLError
from urllib.request import urlopen
health_url = "https://progearsalesai-p2wm.onrender.com/health"
try:
    with urlopen(health_url, timeout=30) as response:
        health = json.load(response)
    print("Demo service status:", health.get("status", "unknown"))
    print("This was a read-only availability check.")
except URLError:
    print("The optional health check could not reach the demo service.")
    print("The integration guide remains fully usable without it.")


## 26. Glossary

- **Authentication:** Proving who a person or system is.
- **Authorization:** Deciding what that identity may do.
- **Workload Principal:** The governed identity representing the AI agent.
- **Direct User access:** User sign-on through the application bound to the registered AI agent.
- **Client registration:** How the AI agent authenticates to Okta.
- **Resource connection:** The maximum downstream resource and scope boundary configured for the agent.
- **Scope:** A narrow permission, such as `inventory:read`.
- **ID-JAG:** A short-lived signed proof binding an agent to the user it represents.
- **Custom Authorization Server:** The policy boundary that issues the final resource access token.
- **Agent Gateway:** A research-release gateway capability that uses the same agent identity to mediate and audit tool calls.

## Final takeaway

You do not replace your custom agent with Okta. You add a governed agent identity, direct user sign-on, explicit resource connections, and a two-step exchange at the point where the agent is about to call protected data. The result is one customer-facing agent whose actions remain attributable to the signed-in user and limited by centrally managed policy.


# Official references

- [Set up AI agent token exchange](https://developer.okta.com/docs/guides/ai-agent-token-exchange/-/main/)
- [Okta for AI Agents API overview](https://developer.okta.com/docs/api/secures-ai)
- [AI agents API](https://developer.okta.com/docs/api/secures-ai/ai-agents)
- [Okta System Log event types](https://developer.okta.com/docs/reference/api/event-types/)
- [System Log query guide](https://developer.okta.com/docs/reference/system-log-query/)
- [ProGear implementation guide](../docs/implementation-guide.md)
- [Platform-neutral token exchange module](../examples/token_exchange.py)

Documentation and product UI evolve. Confirm current field names and supported client methods in your Okta org before production rollout.
